# RSNA Knee Abnormality Detection

## 1. Project Goal

The goal of this project is to build a deep-learning system that takes **knee MRI scans** as input and predicts multiple knee abnormalities.

The abnormalities we are interested in are:

1. ACL
2. MCL
3. Medial Meniscus
4. Lateral Meniscus
5. Medial OA
6. Lateral OA
7. PF OA
8. Effusion
9. Synovitis
10. Baker's
11. Contusion
12. Fracture

This is therefore a **multi-label medical image classification problem**.

A single MRI study can contain multiple abnormalities at the same time.

For example:

```text
Study
 ├── ACL = 1
 ├── MCL = 0
 ├── Medial Meniscus = 1
 ├── Fracture = 0
 └── ...
```

---

# 2. Dataset Structure

The dataset is located at:

```text
/kaggle/input/competitions/rsna-knee-abnormality-detection
```

The important files/directories are:

```text
rsna-knee-abnormality-detection/
│
├── train.csv
├── test.csv
│
├── train_series.csv
├── test_series.csv
│
├── train_series/
│   └── StudyInstanceUID/
│       └── SeriesInstanceUID/
│           ├── image1.dcm
│           ├── image2.dcm
│           ├── image3.dcm
│           └── ...
│
└── test_series/
    └── StudyInstanceUID/
        └── SeriesInstanceUID/
            ├── image1.dcm
            ├── image2.dcm
            └── ...
```

---

# 3. CSV Files

## 3.1 `train.csv`

The training CSV contains:

```text
StudyInstanceUID
Report
ACL
MCL
Medial Meniscus
Lateral Meniscus
Medial OA
Lateral OA
PF OA
Effusion
Synovitis
Baker's
Contusion
Fracture
```

### Meaning

`StudyInstanceUID` identifies one MRI study.

`Report` contains the radiologist's report for that study.

The remaining columns represent abnormality labels.

---

## 3.2 `train_series.csv`

The series CSV contains:

```text
StudyInstanceUID
SeriesInstanceUID
Fluid_Sensitive
Fat_Suppression
Anatomical_Plane
```

This provides information about the MRI series belonging to each study.

For example:

```text
StudyInstanceUID
        │
        ├── SeriesInstanceUID 1
        │      └── Sagittal MRI
        │
        ├── SeriesInstanceUID 2
        │      └── Coronal MRI
        │
        └── SeriesInstanceUID 3
               └── Axial MRI
```

---

# 4. Important Dataset Discovery

We inspected the training CSV and found:

```text
Total studies: 4407

Studies with ALL labels NaN: 4349

Studies with at least one label: 58

Percentage with ALL labels NaN:
98.68%
```

This is the most important issue discovered so far.

Approximately **98.7% of the studies do not have structured labels in the abnormality columns**.

However, these studies are NOT useless.

They still contain:

* MRI/DICOM images
* MRI series information
* Radiology reports

Therefore, we will **NOT drop these studies**.

---

# 5. Why We Cannot Simply Convert NaN to 0

We initially considered the possibility of:

```text
NaN → 0
```

but this would be dangerous.

For example:

```text
ACL = NaN
```

does not necessarily mean:

```text
ACL = Normal
```

It could mean:

```text
ACL information is missing
```

Therefore, we will distinguish:

```text
0  = Normal / negative
1  = Abnormal / positive
-1 = Unknown
```

We will not automatically convert unknown labels into negative labels.

---

# 6. Why We Will Not Drop the 4,349 Studies

Dropping the studies would leave:

```text
4407 studies
      ↓
remove 4349
      ↓
58 studies
```

58 studies are not enough for a useful medical image classification model.

Instead, we want to make use of the information available in the other 4,349 studies.

The major source of information is the:

```text
Report
```

column.

---

# 7. Chosen Approach

We will use:

## Report-Based Pseudo-Labeling + MRI Model Training

The overall pipeline will be:

```text
                         4407 STUDIES
                              │
              ┌───────────────┴────────────────┐
              │                                │
          MRI / DICOM                        Report
              │                                │
              │                                ▼
              │                        NLP processing
              │                                │
              │                         Label extraction
              │                                │
              │                         0 / 1 / Unknown
              │                                │
              └───────────────┬────────────────┘
                              │
                              ▼
                    Prepared Training Dataset
                              │
                              ▼
                       3D MRI Volumes
                              │
                              ▼
                     Multi-Label Deep Model
                              │
                              ▼
                   12 Abnormality Predictions
```

---

# 8. Stage 1 — Dataset Understanding

## Status: IN PROGRESS / MOSTLY COMPLETE

We have already:

* Located the dataset.
* Loaded `train.csv`.
* Loaded `train_series.csv`.
* Inspected their columns.
* Inspected the `train_series` directory.
* Confirmed the directory hierarchy.
* Confirmed that DICOM files exist.
* Confirmed that each study contains multiple series.
* Confirmed that each series contains multiple DICOM files.

The directory structure is:

```text
train_series/
    StudyInstanceUID/
        SeriesInstanceUID/
            *.dcm
```

---

# 9. DICOM Structure

A DICOM file represents an individual MRI slice.

For example:

```text
slice1.dcm
slice2.dcm
slice3.dcm
...
slice30.dcm
```

Each slice can contain an image such as:

```text
512 × 512
```

Multiple slices can be stacked:

```text
30 × 512 × 512
```

This gives us a 3D MRI volume.

Conceptually:

```text
2D Slice
    +
2D Slice
    +
2D Slice
    +
...
    ↓
3D MRI Volume
```

Therefore, the dataset contains **2D DICOM slices that can be reconstructed into 3D volumes**.

---

# 10. DICOM Loading Pipeline

For each study:

```text
StudyInstanceUID
        │
        ▼
Find all SeriesInstanceUIDs
        │
        ▼
Find DICOM files
        │
        ▼
Read DICOM metadata
        │
        ▼
Sort slices
        │
        ▼
Read pixel arrays
        │
        ▼
Stack slices
        │
        ▼
3D NumPy volume
```

Example:

```text
30 DICOM slices
       ↓
30 × 512 × 512
       ↓
3D MRI volume
```

---

# 11. Series Information

`train_series.csv` provides:

```text
Fluid_Sensitive
Fat_Suppression
Anatomical_Plane
```

These fields are important because a study can contain multiple MRI series.

For example:

```text
Study
 │
 ├── Series A
 │      └── Sagittal
 │
 ├── Series B
 │      └── Coronal
 │
 └── Series C
        └── Axial
```

We should therefore **not blindly combine every series together**.

We will investigate the available series and determine which series/sequences are appropriate for the model.

---

# 12. Stage 2 — Report-Based Label Extraction

## Status: NEXT STEP

The 4,349 studies contain radiology reports.

Example:

```text
Técnica: RMN de la rodilla.
Resultados: Rotura...
```

Other reports are in different languages.

We have observed reports in languages including:

* English
* Spanish
* German
* French
* Russian
* Turkish

Therefore, simple English keyword matching is not sufficient.

---

# 13. Report → Labels

We want to transform:

```text
Radiology Report
        │
        ▼
NLP Model / Label Extraction
        │
        ▼
12 abnormality labels
```

For example:

```text
"ACL rupture"
```

could produce:

```text
ACL = 1
```

While:

```text
"ACL is intact"
```

could produce:

```text
ACL = 0
```

If the report does not provide enough information:

```text
ACL = -1
```

where:

```text
-1 = Unknown
```

---

# 14. Negation Handling

This is extremely important.

The presence of a word does not necessarily mean that the abnormality exists.

For example:

```text
"No evidence of ACL tear."
```

contains:

```text
ACL
tear
```

but the correct label is:

```text
ACL = 0
```

Therefore, the report-processing system must understand **negation and context**.

---

# 15. Multilingual Processing

Because the reports are multilingual, the planned approach is:

```text
English report
Spanish report
German report
French report
Russian report
Turkish report
       │
       ▼
Multilingual NLP
       │
       ▼
Normalized medical meaning
       │
       ▼
Abnormality labels
```

We will avoid creating a huge manually maintained dictionary unless it is necessary.

---

# 16. Validation of Pseudo-Labels

We have 58 studies that contain structured labels.

These will be extremely important.

We can use them as a validation/reference set:

```text
                 58 labeled studies
                         │
              ┌──────────┴──────────┐
              │                     │
        Structured labels       NLP labels
              │                     │
              └──────────┬──────────┘
                         │
                         ▼
                      Compare
                         │
                         ▼
               Precision / Recall
                       / F1
```

Example:

```text
             Actual    NLP
ACL            1        1       ✓
MCL            0        0       ✓
Fracture       1        0       ✗
```

We will evaluate the report-labeling method before using it on all 4,349 unlabeled studies.

---

# 17. Confidence-Based Pseudo-Labels

We should not blindly trust every generated label.

Instead:

```text
Report
  │
  ▼
Label extraction
  │
  ├── High confidence
  │       ↓
  │    Use label
  │
  ├── Medium confidence
  │       ↓
  │    Consider separately
  │
  └── Low confidence
          ↓
       Unknown
```

This helps prevent noisy labels from damaging the MRI model.

---

# 18. Target Dataset

Eventually, we want to create a prepared dataset similar to:

```text
StudyInstanceUID
SeriesInstanceUID
ACL
MCL
Medial Meniscus
Lateral Meniscus
Medial OA
Lateral OA
PF OA
Effusion
Synovitis
Baker's
Contusion
Fracture
```

Example:

```text
Study001    Series001    1    0    1    0    ...
Study002    Series002    0    1   -1    0    ...
Study003    Series003   -1   -1    0    1    ...
```

Where:

```text
0  = Negative
1  = Positive
-1 = Unknown
```

---

# 19. Stage 3 — MRI Preprocessing

## Status: NOT STARTED

Once the labels are prepared, we will process the DICOM images.

The pipeline will be:

```text
DICOM
  │
  ▼
Read pixel data
  │
  ▼
Correct slice ordering
  │
  ▼
Select appropriate MRI series
  │
  ▼
Create 3D volume
  │
  ▼
Normalize
  │
  ▼
Resize / resample
  │
  ▼
Save prepared volume
```

Possible final representation:

```text
Depth × Height × Width
```

For example:

```text
32 × 224 × 224
```

The exact dimensions will be decided after inspecting the actual MRI data.

---

# 20. Study-Level Train/Validation Split

The dataset must be split at the **study level**.

We should NOT randomly split individual MRI slices.

Incorrect:

```text
Study001
 ├── Slice 1 → Train
 ├── Slice 2 → Train
 ├── Slice 3 → Validation
```

This can cause data leakage.

Correct:

```text
Study001 → Train

Study002 → Train

Study003 → Validation
```

All slices belonging to a study stay in the same split.

---

# 21. Stage 4 — Model

## Status: NOT STARTED

After preprocessing, the task becomes:

```text
3D MRI Volume
       │
       ▼
Deep Learning Model
       │
       ▼
Feature Representation
       │
       ▼
12 outputs
```

The output will represent:

```text
ACL
MCL
Medial Meniscus
Lateral Meniscus
Medial OA
Lateral OA
PF OA
Effusion
Synovitis
Baker's
Contusion
Fracture
```

This is a **multi-label classification model**.

---

# 22. Possible Model Architecture

We will decide the exact architecture after understanding the data.

Possible approaches include:

```text
3D CNN
```

or:

```text
2D CNN + slice aggregation
```

or:

```text
CNN + Transformer
```

or:

```text
3D Vision Transformer
```

We will start with a reasonable baseline rather than immediately choosing a very complicated architecture.

---

# 23. Model Output

The model should produce 12 probabilities.

Example:

```text
ACL             0.91
MCL             0.03
Medial Meniscus 0.84
Lateral Meniscus 0.12
Medial OA       0.76
Lateral OA      0.31
PF OA           0.65
Effusion        0.88
Synovitis       0.25
Baker's         0.04
Contusion       0.08
Fracture        0.02
```

These probabilities can then be converted into predictions using appropriate thresholds.

---

# 24. Evaluation

Because this is a multi-label classification problem and medical data can be imbalanced, we will not rely only on accuracy.

We will consider metrics such as:

* ROC-AUC
* PR-AUC
* Precision
* Recall
* F1-score
* Per-class performance

For example:

```text
ACL              F1
MCL              F1
Medial Meniscus  F1
...
Fracture         F1
```

We will also investigate class imbalance.

---

# 25. Final Goal

The final system should be capable of:

```text
                 Knee MRI
                    │
                    ▼
              DICOM processing
                    │
                    ▼
                3D volume
                    │
                    ▼
              Deep Learning
                    │
                    ▼
       ┌─────────────────────────┐
       │ Abnormality Predictions │
       ├─────────────────────────┤
       │ ACL                     │
       │ MCL                     │
       │ Medial Meniscus         │
       │ Lateral Meniscus        │
       │ Medial OA               │
       │ Lateral OA              │
       │ PF OA                   │
       │ Effusion                │
       │ Synovitis               │
       │ Baker's                 │
       │ Contusion               │
       │ Fracture                │
       └─────────────────────────┘
```

---

# 26. Current Progress

| Component                                      | Status                   |
| ---------------------------------------------- | ------------------------ |
| Dataset located                                | ✅ Complete               |
| `train.csv` loaded                             | ✅ Complete               |
| `train_series.csv` loaded                      | ✅ Complete               |
| CSV columns inspected                          | ✅ Complete               |
| DICOM directory inspected                      | ✅ Complete               |
| Study → Series → DICOM relationship identified | ✅ Complete               |
| DICOM slice loading                            | ✅ Initial implementation |
| 3D volume creation                             | ✅ Initial implementation |
| NaN problem identified                         | ✅ Complete               |
| 4,349 unlabeled studies identified             | ✅ Complete               |
| Report availability identified                 | ✅ Complete               |
| Report-based pseudo-labeling                   | ⏳ Next                   |
| Validate pseudo-labels                         | ⏳ Pending                |
| Select MRI series                              | ⏳ Pending                |
| Full DICOM preprocessing                       | ⏳ Pending                |
| Train/validation split                         | ⏳ Pending                |
| Baseline model                                 | ⏳ Pending                |
| Model training                                 | ⏳ Pending                |
| Evaluation                                     | ⏳ Pending                |
| Final inference pipeline                       | ⏳ Pending                |

---

# 27. Immediate Next Steps

We should **not jump to model training yet**.

The next steps are:

```text
STEP 1
Inspect the 58 labeled studies
        ↓
Understand what 0 / 1 / NaN mean


STEP 2
Analyze their reports
        ↓
Determine how reports describe abnormalities


STEP 3
Build report → label extraction
        ↓
Handle multilingual text
        ↓
Handle medical negation


STEP 4
Validate extraction
        ↓
Compare against 58 structured labels


STEP 5
Apply reliable extraction
        ↓
Generate pseudo-labels for 4,349 studies


STEP 6
Inspect train_series.csv
        ↓
Determine useful MRI series


STEP 7
Convert DICOM slices → 3D volumes


STEP 8
Create study-level train/validation split


STEP 9
Train baseline multi-label model


STEP 10
Evaluate and improve
```

---

# 28. Important Rules for This Project

### Rule 1 — Never blindly convert NaN to 0

```text
NaN ≠ automatically 0
```

Use:

```text
-1 = Unknown
```

until the label can be established.

### Rule 2 — Do not discard the 4,349 studies

They contain:

```text
MRI + Reports
```

and therefore potentially valuable training information.

### Rule 3 — Split by study

Never allow slices from the same study to appear in both training and validation sets.

### Rule 4 — Validate pseudo-labels

Pseudo-labels must be checked against the available structured labels before being trusted.

### Rule 5 — Start simple

We will first build a reliable baseline before moving to complex architectures.

---

# 29. Final Project Architecture

The complete project will eventually look like:

```text
                         RSNA DATASET
                              │
            ┌─────────────────┴─────────────────┐
            │                                   │
       train.csv                         train_series.csv
            │                                   │
            │                                   │
       Study + Report                    Study + Series
            │                                   │
            ▼                                   ▼
     Report Processing                    Series Selection
            │                                   │
            ▼                                   ▼
     Pseudo-labels                         DICOM Files
            │                                   │
            └────────────────┬──────────────────┘
                             │
                             ▼
                     Prepared Dataset
                             │
                             ▼
                    3D MRI Preprocessing
                             │
                             ▼
                  Study-Level Data Split
                       /             \
                    Train          Validation
                       \             /
                        \           /
                         ▼         ▼
                       Deep Learning
                             │
                             ▼
                     Multi-label Output
                             │
                             ▼
                  12 Knee Abnormalities
                             │
                             ▼
                       Evaluation
```

---

# 30. Current Decision

### We are currently here:

```text
Dataset
   ↓
Understand CSV
   ↓
Understand DICOM structure
   ↓
Discover 98.68% NaN labels
   ↓
Decide NOT to drop them
   ↓
Use reports to derive reliable pseudo-labels
   ↓
        ← CURRENT STAGE
```

### Our immediate objective:

> **Build and validate a reliable report-to-label pipeline using the 58 structured-label studies, then use it to generate high-confidence pseudo-labels for the remaining 4,349 studies.**

Only after that will we proceed to full MRI preprocessing and model training.

---

# 31. Expected Final Result

At the end of the project, we want a reproducible pipeline:

```text
Raw RSNA Dataset
       │
       ▼
Data Preparation
       │
       ▼
Report Label Extraction
       │
       ▼
DICOM Processing
       │
       ▼
3D MRI Dataset
       │
       ▼
Multi-label Model
       │
       ▼
Evaluation
       │
       ▼
Inference on New Knee MRI
```

The final model should accept a knee MRI study and return the predicted probability of each of the 12 abnormalities.


In [1]:
# ============================================================
# RSNA Knee Abnormality Detection
# Dataset Loading + DICOM -> 3D Volume
# ============================================================

!pip install pydicom -q

In [2]:
import os
import numpy as np
import pandas as pd
import pydicom

In [3]:

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
TRAIN_SERIES_CSV = os.path.join(BASE_PATH, "train_series.csv")

TRAIN_SERIES_PATH = os.path.join(BASE_PATH, "train_series")

# ------------------------------------------------------------
# 2. Check paths
# ------------------------------------------------------------

print("Checking paths...\n")

print("Base path exists:",
      os.path.exists(BASE_PATH))

print("Train CSV exists:",
      os.path.exists(TRAIN_CSV))

print("Train Series CSV exists:",
      os.path.exists(TRAIN_SERIES_CSV))

print("Train Series folder exists:",
      os.path.exists(TRAIN_SERIES_PATH))



Checking paths...

Base path exists: True
Train CSV exists: True
Train Series CSV exists: True
Train Series folder exists: True


In [4]:
# ------------------------------------------------------------
# 3. Load CSV files
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_CSV)

train_series_df = pd.read_csv(TRAIN_SERIES_CSV)

print("\n==============================")
print("TRAIN CSV")
print("==============================")

print("Shape:", train_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_df.head())

print("\n==============================")
print("TRAIN SERIES CSV")
print("==============================")

print("Shape:", train_series_df.shape)

print("\nColumns:")
print(train_series_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_series_df.head())


TRAIN CSV
Shape: (4407, 14)

Columns:
['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

First 5 rows:


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



TRAIN SERIES CSV
Shape: (24371, 5)

Columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']

First 5 rows:


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal


In [5]:
# ------------------------------------------------------------
# 4. Select one study
# ------------------------------------------------------------

study_id = train_df.iloc[0]["StudyInstanceUID"]

print("\n==============================")
print("SELECTED STUDY")
print("==============================")

print("StudyInstanceUID:")
print(study_id)


# ------------------------------------------------------------
# 5. Get labels for this study
# ------------------------------------------------------------

study_row = train_df[
    train_df["StudyInstanceUID"] == study_id
].iloc[0]

print("\n==============================")
print("STUDY LABELS")
print("==============================")

label_columns = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print(study_row[label_columns])


# ------------------------------------------------------------
# 6. Find series belonging to this study
# ------------------------------------------------------------

study_path = os.path.join(
    TRAIN_SERIES_PATH,
    study_id
)

print("\n==============================")
print("STUDY DIRECTORY")
print("==============================")

print(study_path)
print("Exists:", os.path.exists(study_path))

series_ids = [
    x for x in os.listdir(study_path)
    if os.path.isdir(os.path.join(study_path, x))
]

print("\nNumber of series:", len(series_ids))

for series_id in series_ids:
    print(series_id)


# ------------------------------------------------------------
# 7. Get metadata for the series from train_series.csv
# ------------------------------------------------------------

study_series_df = train_series_df[
    train_series_df["StudyInstanceUID"] == study_id
]

print("\n==============================")
print("SERIES INFORMATION")
print("==============================")

display(study_series_df)


# ------------------------------------------------------------
# 8. Function to load one DICOM series
# ------------------------------------------------------------

def load_dicom_series(study_id, series_id):

    series_path = os.path.join(
        TRAIN_SERIES_PATH,
        study_id,
        series_id
    )

    # Get DICOM files
    dicom_files = [
        f for f in os.listdir(series_path)
        if f.lower().endswith(".dcm")
    ]

    slices = []

    for file_name in dicom_files:

        file_path = os.path.join(
            series_path,
            file_name
        )

        ds = pydicom.dcmread(file_path)

        slices.append(ds)

    # Sort slices using InstanceNumber
    slices.sort(
        key=lambda x: int(
            getattr(x, "InstanceNumber", 0)
        )
    )

    # Convert to numpy array
    volume = np.stack(
        [ds.pixel_array for ds in slices]
    )

    # Convert to float32
    volume = volume.astype(np.float32)

    # Normalize to [0, 1]
    min_value = volume.min()
    max_value = volume.max()

    volume = (
        volume - min_value
    ) / (
        max_value - min_value + 1e-8
    )

    return volume, slices


# ------------------------------------------------------------
# 9. Load the first series
# ------------------------------------------------------------

if len(series_ids) > 0:

    selected_series_id = series_ids[0]

    print("\n==============================")
    print("SELECTED SERIES")
    print("==============================")

    print("SeriesInstanceUID:")
    print(selected_series_id)

    volume, slices = load_dicom_series(
        study_id,
        selected_series_id
    )

    print("\n==============================")
    print("VOLUME INFORMATION")
    print("==============================")

    print("Number of slices:", len(slices))

    print("Volume shape:", volume.shape)

    print("Volume dtype:", volume.dtype)

    print("Minimum value:", volume.min())

    print("Maximum value:", volume.max())

else:

    print("No series found for this study.")


SELECTED STUDY
StudyInstanceUID:
1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260

STUDY LABELS
ACL                 NaN
MCL                 NaN
Medial Meniscus     NaN
Lateral Meniscus    NaN
Medial OA           NaN
Lateral OA          NaN
PF OA               NaN
Effusion            NaN
Synovitis           NaN
Baker's             NaN
Contusion           NaN
Fracture            NaN
Name: 0, dtype: object

STUDY DIRECTORY
/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series/1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260
Exists: True

Number of series: 5
1.2.826.0.1.3680043.8.498.13821229744997220641575291927426543265
1.2.826.0.1.3680043.8.498.75714899997203615784077798038670363546
1.2.826.0.1.3680043.8.498.40734206102458723096154687147390476697
1.2.826.0.1.3680043.8.498.23084836536722595275828690293168736174
1.2.826.0.1.3680043.8.498.12343110195036213483454091715412333772

SERIES INFORMATION


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal



SELECTED SERIES
SeriesInstanceUID:
1.2.826.0.1.3680043.8.498.13821229744997220641575291927426543265

VOLUME INFORMATION
Number of slices: 18
Volume shape: (18, 560, 560)
Volume dtype: float32
Minimum value: 0.0
Maximum value: 1.0


In [6]:

# Studies where ALL labels are NaN
all_nan = train_df[label_columns].isna().all(axis=1)

# Studies where at least one label exists
some_labels = train_df[label_columns].notna().any(axis=1)

print("Total studies:", len(train_df))

print("Studies with ALL labels NaN:", all_nan.sum())

print("Studies with at least one label:", some_labels.sum())

print(
    "Percentage with ALL labels NaN:",
    all_nan.mean() * 100,
    "%"
)

nan_studies = train_df[all_nan]

display(
    nan_studies[
        ["StudyInstanceUID", "Report"]
    ].head(10)
)

Total studies: 4407
Studies with ALL labels NaN: 4349
Studies with at least one label: 58
Percentage with ALL labels NaN: 98.68391195824825 %


,StudyInstanceUID,Report
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no..."
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...
5,1.2.826.0.1.3680043.8.498.10018552945042470316...,Antecedentes Clínicos:\nCondromalacia rotulian...
6,1.2.826.0.1.3680043.8.498.10018749256316287542...,"MRI of left knee with -Locator, SG PD FatSat, ..."
7,1.2.826.0.1.3680043.8.498.10021179701366615563...,"МР находка: МР данни за ставен излив. Костите,..."
8,1.2.826.0.1.3680043.8.498.10021288557171866794...,"SOL DİZ MRG. Tetkik protokolü: Çok düzlemli, ç..."
9,1.2.826.0.1.3680043.8.498.10025765742726180988...,"MRI of left Knee with \n-Locator, SG PD FatSat..."
